In [1]:
!pip install transformers
!pip install requests

In [2]:
import requests
import json
import pandas as pd

# Step 1: Search for a school and get its ID
def get_school_id(school_name):
    url = "https://www.ratemyprofessors.com/graphql"
    
    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.ratemyprofessors.com/",
    }
    
    query = """
    query SchoolSearchQuery($query: SchoolSearchQuery!) {
        newSearch {
            schools(query: $query) {
                edges {
                    node {
                        id
                        name
                        city
                        state
                    }
                }
            }
        }
    }
    """
    
    payload = {"query": query, "variables": {"query": {"text": school_name}}}
    response = requests.post(url, json=payload, headers=headers)
    
    print("Status:", response.status_code)  # add this temporarily
    data = response.json()
    
    if "errors" in data:
        print("GraphQL errors:", data["errors"])
        return []
    
    schools = data["data"]["newSearch"]["schools"]["edges"]
    for school in schools:
        print(school["node"])
    
    return schools


results = get_school_id("Purdue University West Lafayette")

Status: 200
{'city': 'West Lafayette', 'id': 'U2Nob29sLTc4Mw==', 'name': 'Purdue University - West Lafayette', 'state': 'IN'}
{'city': 'West Lafayette', 'id': 'U2Nob29sLTE3NTk5', 'name': 'Purdue University Global', 'state': 'IN'}
{'city': 'Hammond', 'id': 'U2Nob29sLTc4NA==', 'name': 'Purdue University Northwest', 'state': 'IN'}
{'city': 'Kokomo', 'id': 'U2Nob29sLTEzMzMw', 'name': 'Purdue University - Kokomo', 'state': 'IN'}
{'city': 'Columbus', 'id': 'U2Nob29sLTQzOA==', 'name': 'Indiana University - Purdue University', 'state': 'IN'}
{'city': '', 'id': 'U2Nob29sLTc4NQ==', 'name': 'Purdue University Northwest', 'state': ''}
{'city': 'Glendale', 'id': 'U2Nob29sLTEzNDgz', 'name': 'Arizona State University - West', 'state': 'AZ'}
{'city': 'Fort Wayne', 'id': 'U2Nob29sLTg0NTg=', 'name': 'Purdue University - Fort Wayne', 'state': 'IN'}


In [16]:
# Step 2: Get professors at that school
def get_professors(school_id, num_professors=1000):
    url = "https://www.ratemyprofessors.com/graphql"
    
    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.ratemyprofessors.com/",
    }
    
    query = """
    query TeacherSearchQuery($schoolID: ID!, $count: Int!) {
        newSearch {
            teachers(query: {schoolID: $schoolID}, first: $count) {
                edges {
                    node {
                        id
                        firstName
                        lastName
                        department
                        avgRating
                        avgDifficulty
                        numRatings
                    }
                }
            }
        }
    }
    """
    
    payload = {
        "query": query,
        "variables": {"schoolID": school_id, "count": num_professors}
    }
    
    response = requests.post(url, json=payload, headers=headers)
    data = response.json()

    if "errors" in data:
        print("GraphQL errors:", data["errors"])
        return pd.DataFrame()
    
    professors = []
    for edge in data["data"]["newSearch"]["teachers"]["edges"]:
        professors.append(edge["node"])
    
    return pd.DataFrame(professors)


# Use the school ID you got from Step 1
df_profs = get_professors("U2Nob29sLTc4Mw==", num_professors=500)
print(df_profs.head())

   avgDifficulty  avgRating                      department  firstName  \
0            1.8        4.8  Computer & Informational Tech.     Khalil   
1            1.8        4.6                    Anthropology  Elizabeth   
2            2.3        5.0                     Mathematics       Nick   
3            3.3        4.7                       Chemistry  Christina   
4            3.5        4.2           Aerospace Engineering   Kenshiro   

                     id lastName  numRatings  
0  VGVhY2hlci0yOTI0NTMw   Breidi          29  
1  VGVhY2hlci0yMDUyNDQ2    Brite          24  
2  VGVhY2hlci0zMTIzNDU3     Lohr          23  
3  VGVhY2hlci0yMzU2MDYy       Li          17  
4  VGVhY2hlci0yNzQwMDA4    Oguri           6  


In [17]:
# Step 3: Get reviews for a specific professor
def get_reviews(professor_id, num_reviews=20):
    url = "https://www.ratemyprofessors.com/graphql"
    
    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.ratemyprofessors.com/",
    }
    
    query = """
    query RatingsListQuery($id: ID!, $count: Int!) {
        node(id: $id) {
            ... on Teacher {
                firstName
                lastName
                ratings(first: $count) {
                    edges {
                        node {
                            comment
                            qualityRating
                            difficultyRatingRounded
                            date
                            class
                            thumbsUpTotal
                            thumbsDownTotal
                        }
                    }
                }
            }
        }
    }
    """
    
    payload = {
        "query": query,
        "variables": {"id": professor_id, "count": num_reviews}
    }
    
    response = requests.post(url, json=payload, headers=headers)
    data = response.json()

    if "errors" in data:
        print("GraphQL errors:", data["errors"])
        return pd.DataFrame()

    node = data["data"]["node"]
    if node is None:
        print(f"No data found for professor ID: {professor_id}")
        return pd.DataFrame()

    reviews = []
    for edge in node["ratings"]["edges"]:
        r = edge["node"]
        r["professor"] = f"{node['firstName']} {node['lastName']}"
        reviews.append(r)
    
    return pd.DataFrame(reviews)


# Use any professor ID from your df_profs dataframe
df_reviews = get_reviews(df_profs["id"][0])
print(df_reviews.head())

     class                                            comment  \
0  MFET103  He genuinely wants everyone to succeed and mak...   
1  MFET163  What I really liked is that he won't just give...   
2  CNIT175  Take Khalil's lab section if you're worried ab...   
3  MFET163  If you are intimidated by complex software or ...   
4  CNIT175  Always shows up and answers questions. Slide s...   

                            date  difficultyRatingRounded  qualityRating  \
0  2026-05-05 19:48:45 +0000 UTC                        1              5   
1  2026-05-01 20:12:27 +0000 UTC                        2              5   
2  2026-04-09 16:15:19 +0000 UTC                        2              5   
3  2026-04-05 21:28:43 +0000 UTC                        3              5   
4  2025-05-05 16:42:17 +0000 UTC                        3              4   

   thumbsDownTotal  thumbsUpTotal      professor  
0                0              0  Khalil Breidi  
1                0              0  Khalil Breidi  

In [18]:
from tqdm import tqdm
all_reviews = []

for _, prof in tqdm(df_profs.iterrows(), total=len(df_profs)):
    # skip professors with very few reviews
    if prof["numRatings"] < 5:
        continue
    
    try:
        reviews = get_reviews(prof["id"], num_reviews=20)
        reviews["department"] = prof["department"]
        reviews["avgRating"] = prof["avgRating"]
        all_reviews.append(reviews)
    except Exception as e:
        print(f"Failed for {prof['firstName']}: {e}")
        continue

df_all = pd.concat(all_reviews, ignore_index=True)
df_all.to_csv("rmp_reviews.csv", index=False)
print(f"Collected {len(df_all)} reviews across {len(df_profs)} professors")

100%|██████████| 500/500 [07:32<00:00,  1.10it/s]

Collected 5960 reviews across 500 professors


In [19]:
import pandas as pd

df = pd.read_csv("rmp_reviews.csv")
print(df.shape)
print(df.head())

df = df.dropna(subset=["comment"])
df = df[df["comment"].str.len() > 10]
print(f"Clean reviews: {len(df)}")

(5960, 10)
     class                                            comment  \
0  MFET103  He genuinely wants everyone to succeed and mak...   
1  MFET163  What I really liked is that he won't just give...   
2  CNIT175  Take Khalil's lab section if you're worried ab...   
3  MFET163  If you are intimidated by complex software or ...   
4  CNIT175  Always shows up and answers questions. Slide s...   

                            date  difficultyRatingRounded  qualityRating  \
0  2026-05-05 19:48:45 +0000 UTC                        1              5   
1  2026-05-01 20:12:27 +0000 UTC                        2              5   
2  2026-04-09 16:15:19 +0000 UTC                        2              5   
3  2026-04-05 21:28:43 +0000 UTC                        3              5   
4  2025-05-05 16:42:17 +0000 UTC                        3              4   

   thumbsDownTotal  thumbsUpTotal      professor  \
0                0              0  Khalil Breidi   
1                0              0  Kh

In [20]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 1918.89it/s]


In [21]:
aspects = [
    "harsh grading",
    "great teaching",
    "poor teaching",
    "engaging lectures",
    "boring lectures",
    "heavy workload",
    "approachable and kind",
    "unavailable or unhelpful"
]

In [22]:
sample = df["comment"].iloc[0]
print("Review:", sample)

result = classifier(sample, aspects, multi_label=True)

for label, score in zip(result["labels"], result["scores"]):
    bar = "█" * int(score * 20)
    print(f"{label:<25} {bar} {score:.2%}")

Review: He genuinely wants everyone to succeed and makes the lab super low-stress
approachable and kind     ███████████████████ 99.95%
great teaching            ███████████████ 75.24%
engaging lectures          4.93%
heavy workload             0.04%
unavailable or unhelpful   0.02%
poor teaching              0.02%
boring lectures            0.01%
harsh grading              0.01%


In [23]:
from tqdm import tqdm
tqdm.pandas()

def classify_review(comment):
    try:
        result = classifier(comment, aspects, multi_label=True)
        # return a dict of {aspect: score}
        return dict(zip(result["labels"], result["scores"]))
    except Exception as e:
        return {}

# This will take several minutes
df["aspect_scores"] = df["comment"].progress_apply(classify_review)

100%|██████████| 5928/5928 [34:54:50<00:00, 21.20s/it]       


In [24]:
scores_df = pd.json_normalize(df["aspect_scores"])
df = pd.concat([df, scores_df], axis=1)

df.to_csv("rmp_reviews_scored.csv", index=False)
print("Done! Saved scored reviews.")

Done! Saved scored reviews.


In [25]:
aspect_cols = aspects  # the list you defined earlier

professor_profiles = df.groupby("professor")[aspect_cols].mean().reset_index()
print(professor_profiles.head())

             professor  harsh grading  great teaching  poor teaching  \
0        Abigail Banan       0.140377        0.688949       0.240145   
1          Adam Bodony       0.005810        0.904044       0.003711   
2      Adeola Mobolaji       0.010391        0.755176       0.092871   
3      Adrian Wolanski       0.075811        0.908584       0.006604   
4  Alexander Kildishev       0.365222        0.261311       0.684670   

   engaging lectures  boring lectures  heavy workload  approachable and kind  \
0           0.667332         0.108251        0.360218               0.625269   
1           0.599194         0.000180        0.076240               0.921613   
2           0.582555         0.004288        0.191056               0.750608   
3           0.802587         0.005535        0.213911               0.902287   
4           0.273933         0.290611        0.366778               0.221037   

   unavailable or unhelpful  
0                  0.011392  
1                  0.00036